# Лаборатори 5: Пропозицийн логик
Үнэний хүснэгт, CNF, резолюц ба DPLL хэрэгжүүлэлт (AIMA 7-р бүлэг)

## Алхам 1: Өгүүлбэрийг илэрхийлэх

**Шалгах:** (P ∧ Q) ⇒ R өгүүлбэрийг үүсгээд symbols нь {P, Q, R} буцаахыг, to_str
нь зөв хаалттай мөр хэвлэхийг шалгана. Өөр хоёр өгүүлбэрийг гараар үүсгэж
туршина.

In [1]:
import itertools, random

def Sym(n):     return ('sym', n)
def Not(a):     return ('not', a)
def And(a, b):  return ('and', a, b)
def Or(a, b):   return ('or', a, b)
def Imp(a, b):  return ('imp', a, b)
def Iff(a, b):  return ('iff', a, b)

def symbols(s):
    """Өгүүлбэрт орсон тэмдэгтүүдийн нэрийн олонлог (рекурсив)."""
    if s[0] == 'sym':
        return {s[1]}
    result = set()
    for sub in s[1:]:
        result |= symbols(sub)
    return result

OPS = {'and': '&', 'or': '|', 'imp': '=>', 'iff': '<=>'}

def to_str(s):
    """Өгүүлбэрийг хаалттай, уншигдахуйц мөр болгоно."""
    if s[0] == 'sym':
        return s[1]
    if s[0] == 'not':
        return '~' + to_str(s[1])
    return '(' + to_str(s[1]) + ' ' + OPS[s[0]] + ' ' + to_str(s[2]) + ')'

P, Q, R = Sym('P'), Sym('Q'), Sym('R')
s1 = Imp(And(P, Q), R)
print('symbols:', sorted(symbols(s1)))
print('to_str :', to_str(s1))
s2 = Iff(Not(P), Or(Q, Not(R)))
s3 = Or(And(P, Not(Q)), Imp(R, P))
for s in (s2, s3):
    print(to_str(s), sorted(symbols(s)))

symbols: ['P', 'Q', 'R']
to_str : ((P & Q) => R)
(~P <=> (Q | ~R)) ['P', 'Q', 'R']
((P & ~Q) | (R => P)) ['P', 'Q', 'R']


## Алхам 2: Семантик

**Шалгах:** m1 = {P: True, Q: False, R: True} загварт (P ∧ Q) ⇒ R үнэн, P ⇔ Q худал,
¬P ∨ Q худал гарна. Гараар бодсон хариутай тулгана.


In [2]:
def pl_true(s, model):
    op = s[0]
    if op == 'sym':
        return model[s[1]]
    if op == 'not':
        return not pl_true(s[1], model)
    if op == 'and':
        return pl_true(s[1], model) and pl_true(s[2], model)
    if op == 'or':
        return pl_true(s[1], model) or pl_true(s[2], model)
    if op == 'imp':
        return (not pl_true(s[1], model)) or pl_true(s[2], model)
    if op == 'iff':
        return pl_true(s[1], model) == pl_true(s[2], model)
    raise ValueError('Тодорхойгүй холбоос: ' + str(op))

m1 = {'P': True, 'Q': False, 'R': True}
print('(P & Q) => R :', pl_true(Imp(And(P, Q), R), m1))   # True
print('P <=> Q      :', pl_true(Iff(P, Q), m1))           # False
print('~P | Q       :', pl_true(Or(Not(P), Q), m1))       # False

(P & Q) => R : True
P <=> Q      : False
~P | Q       : False


## Алхам 3: Үнэний хүснэгт ба логик дүгнэлт

**Шалгах:** Дараах дүгнэлтүүд зөв гарах ёстой.
(P ∧ (P ⇒ Q)) ⊨ Q нь True (Modus Ponens). (P ∨ Q) ⊨ P нь False.
(¬Q ∧ (P ⇒ Q)) ⊨ ¬P нь True (Modus Tollens).
P ∨ ¬P нь valid, P ∧ ¬P нь unsatisfiable.
n тэмдэгттэй үед үнэн дүгнэлт яг 2ⁿ загвар шалгана.


In [3]:
def all_models(syms):
    syms = sorted(syms)
    for values in itertools.product([True, False], repeat=len(syms)):
        yield dict(zip(syms, values))

def conjoin(sentences):
    """Олон өгүүлбэрийг And-аар нэг өгүүлбэр болгоно."""
    if isinstance(sentences, tuple):
        return sentences
    result = sentences[0]
    for s in sentences[1:]:
        result = And(result, s)
    return result

def tt_entails(kb, alpha):
    """KB |= alpha эсэх. (үр дүн, шалгасан загварын тоо) буцаана."""
    kb = conjoin(kb)
    count = 0
    for model in all_models(symbols(kb) | symbols(alpha)):
        count += 1
        if pl_true(kb, model) and not pl_true(alpha, model):
            return False, count
    return True, count

def is_valid(s):
    return all(pl_true(s, m) for m in all_models(symbols(s)))

def is_satisfiable(s):
    return any(pl_true(s, m) for m in all_models(symbols(s)))

def print_truth_table(*sents):
    syms = sorted(set().union(*(symbols(s) for s in sents)))
    header = syms + [to_str(s) for s in sents]
    widths = [max(5, len(h)) for h in header]
    print(' | '.join(h.ljust(w) for h, w in zip(header, widths)))
    print('-+-'.join('-' * w for w in widths))
    for m in all_models(syms):
        row = [m[x] for x in syms] + [pl_true(s, m) for s in sents]
        print(' | '.join(('T' if v else 'F').ljust(w) for v, w in zip(row, widths)))

print('Modus Ponens  :', tt_entails(And(P, Imp(P, Q)), Q))
print('(P|Q) |= P    :', tt_entails(Or(P, Q), P))
print('Modus Tollens :', tt_entails(And(Not(Q), Imp(P, Q)), Not(P)))
print('P|~P valid    :', is_valid(Or(P, Not(P))))
print('P&~P satisf.  :', is_satisfiable(And(P, Not(P))))
print()
print_truth_table(Not(And(P, Q)), Or(Not(P), Not(Q)))

Modus Ponens  : (True, 4)
(P|Q) |= P    : (False, 3)
Modus Tollens : (True, 4)
P|~P valid    : True
P&~P satisf.  : False

P     | Q     | ~(P & Q) | (~P | ~Q)
------+-------+----------+----------
T     | T     | F        | F        
T     | F     | T        | T        
F     | T     | T        | T        
F     | F     | T        | T        


## Алхам 4: CNF-д хөрвүүлэх

**Шалгах:** B ⇔ (A ∨ C) өгүүлбэрийн clause нь яг гурав байна: {¬B, A, C}, {¬A, B},
{¬C, B}.
to_cnf үр дүн нь анхны өгүүлбэртэй бүх загварт ижил утгатай эсэхийг Алхам 3-
ын функцээр шалгана. Өөр гурван өгүүлбэр дээр мөн ингэж шалгана.

In [4]:
def eliminate_iff(s):
    if s[0] == 'sym':
        return s
    if s[0] == 'not':
        return Not(eliminate_iff(s[1]))
    a, b = eliminate_iff(s[1]), eliminate_iff(s[2])
    if s[0] == 'iff':
        return And(Imp(a, b), Imp(b, a))
    return (s[0], a, b)

def eliminate_imp(s):
    if s[0] == 'sym':
        return s
    if s[0] == 'not':
        return Not(eliminate_imp(s[1]))
    a, b = eliminate_imp(s[1]), eliminate_imp(s[2])
    if s[0] == 'imp':
        return Or(Not(a), b)
    return (s[0], a, b)

def push_not(s):
    if s[0] == 'sym':
        return s
    if s[0] == 'not':
        t = s[1]
        if t[0] == 'sym':
            return s
        if t[0] == 'not':                       # ~~a = a
            return push_not(t[1])
        if t[0] == 'and':                       # ~(a&b) = ~a | ~b
            return Or(push_not(Not(t[1])), push_not(Not(t[2])))
        if t[0] == 'or':                        # ~(a|b) = ~a & ~b
            return And(push_not(Not(t[1])), push_not(Not(t[2])))
        raise ValueError('imp/iff эхлээд арилгах ёстой')
    return (s[0], push_not(s[1]), push_not(s[2]))

def distribute(s):
    if s[0] in ('sym', 'not'):
        return s
    a, b = distribute(s[1]), distribute(s[2])
    if s[0] == 'and':
        return And(a, b)
    # s[0] == 'or'
    if a[0] == 'and':                           # (x&y)|b = (x|b)&(y|b)
        return And(distribute(Or(a[1], b)), distribute(Or(a[2], b)))
    if b[0] == 'and':                           # a|(x&y) = (a|x)&(a|y)
        return And(distribute(Or(a, b[1])), distribute(Or(a, b[2])))
    return Or(a, b)

def to_cnf(s):
    return distribute(push_not(eliminate_imp(eliminate_iff(s))))

def negate_lit(l):
    return l[1:] if l.startswith('~') else '~' + l

def _conjuncts(s):
    return _conjuncts(s[1]) + _conjuncts(s[2]) if s[0] == 'and' else [s]

def _disjuncts(s):
    return _disjuncts(s[1]) + _disjuncts(s[2]) if s[0] == 'or' else [s]

def _lit(s):
    return s[1] if s[0] == 'sym' else '~' + s[1][1]

def is_tautology(clause):
    return any(negate_lit(l) in clause for l in clause)

def to_clauses(s):
    """CNF өгүүлбэр -> frozenset clause-үүдийн set (tautology хасна)."""
    clauses = set()
    for c in _conjuncts(to_cnf(s)):
        clause = frozenset(_lit(l) for l in _disjuncts(c))
        if not is_tautology(clause):
            clauses.add(clause)
    return clauses

def clause_str(c):
    return '{' + ', '.join(sorted(c, key=lambda l: (l.lstrip('~'), l))) + '}' if c else '{}'

A, B, C = Sym('A'), Sym('B'), Sym('C')
bac = Iff(B, Or(A, C))
print('CNF     :', to_str(to_cnf(bac)))
print('Clauses :', sorted(clause_str(c) for c in to_clauses(bac)))

def equivalent(s, t):
    return is_valid(Iff(s, t))

tests = [bac, Imp(And(P, Q), R), Not(Iff(P, Or(Q, R))), Or(And(P, Q), And(Not(P), R))]
for s in tests:
    print(f'{to_str(s):32s} ≡ CNF : {equivalent(s, to_cnf(s))}')

CNF     : ((~B | (A | C)) & ((~A | B) & (~C | B)))
Clauses : ['{A, ~B, C}', '{B, ~C}', '{~A, B}']
(B <=> (A | C))                  ≡ CNF : True
((P & Q) => R)                   ≡ CNF : True
~(P <=> (Q | R))                 ≡ CNF : True
((P & Q) | (~P & R))             ≡ CNF : True


## Алхам 5: Резолюц

**Шалгах:** resolve({A, B}, {¬A, C}) нь {B, C} буцаана. resolve({C}, {¬C})
нь хоосон clause буцаана.
KB: (A ∨ B), (¬A ∨ C), (¬B ∨ C) үед KB ⊨ C нь True, KB ⊨ A нь False.
Тестийн бүх тохиолдолд pl_resolution-ий хариу Алхам 3-ын tt_entails-тэй
таарах ёстой.


In [5]:
RES_COUNT = 0

def resolve(ci, cj):
    resolvents = []
    for l in ci:
        if negate_lit(l) in cj:
            new = (ci - {l}) | (cj - {negate_lit(l)})
            if not is_tautology(new):
                resolvents.append(frozenset(new))
    return resolvents

def pl_resolution(kb_clauses, alpha, limit=100_000, negate=True):
    """KB |= alpha эсэхийг refutation-аар шалгана.
    negate=False бол alpha-г үгүйсгэлгүй нэмнэ (satisfiability шалгахад)."""
    global RES_COUNT
    RES_COUNT = 0
    extra = to_clauses(Not(alpha)) if negate else (to_clauses(alpha) if alpha else set())
    clauses = set(kb_clauses) | extra
    while True:
        new = set()
        cl = sorted(clauses, key=clause_str)
        for i in range(len(cl)):
            for j in range(i + 1, len(cl)):
                for r in resolve(cl[i], cl[j]):
                    RES_COUNT += 1
                    if not r:
                        return True
                    if RES_COUNT > limit:
                        return None             # "хэт удаан"
                    new.add(r)
        if new <= clauses:
            return False
        clauses |= new

print('resolve({A,B},{~A,C}) =', [clause_str(c) for c in resolve(frozenset({'A','B'}), frozenset({'~A','C'}))])
print('resolve({C},{~C})     =', [clause_str(c) for c in resolve(frozenset({'C'}), frozenset({'~C'}))])

kb5 = [Or(A, B), Or(Not(A), C), Or(Not(B), C)]
kb5_clauses = set().union(*(to_clauses(s) for s in kb5))
for alpha in (C, A):
    r = pl_resolution(kb5_clauses, alpha)
    t, _ = tt_entails(kb5, alpha)
    print(f'KB |= {to_str(alpha)} : resolution={r}, resolvents={RES_COUNT}, tt_entails={t}')

resolve({A,B},{~A,C}) = ['{B, C}']
resolve({C},{~C})     = ['{}']
KB |= C : resolution=True, resolvents=25, tt_entails=True
KB |= A : resolution=False, resolvents=17, tt_entails=False


## Алхам 6: DPLL

**Шалгах:** (A), (¬A ∨ B), (¬B ∨ C), (¬C ∨ D ∨ E), (¬D ∨ ¬E) нь satisfiable. Загварт A,
B, C үнэн байх ба D, E-ийн яг нэг нь үнэн байна.
(A ∨ B), (¬A ∨ B), (A ∨ ¬B), (¬A ∨ ¬B) нь unsatisfiable (False буцаана).
KB ⊨ α бол CNF(KB ∧ ¬α) нь unsatisfiable. Алхам 5-ын KB ба C дээр үүнийг
шалгана.

In [6]:
DPLL_CALLS = 0

def lit_value(l, model):
    v = model.get(l.lstrip('~'))
    if v is None:
        return None
    return (not v) if l.startswith('~') else v

def clause_value(clause, model):
    unknown = False
    for l in clause:
        v = lit_value(l, model)
        if v is True:
            return True
        if v is None:
            unknown = True
    return None if unknown else False

def find_pure_symbol(symbols, clauses, model):
    open_clauses = [c for c in clauses if clause_value(c, model) is not True]
    for s in symbols:
        pos = any(s in c for c in open_clauses)
        neg = any('~' + s in c for c in open_clauses)
        if pos != neg:
            return s, pos
    return None

def find_unit_clause(clauses, model):
    for c in clauses:
        if clause_value(c, model) is not None:
            continue
        unassigned = [l for l in c if lit_value(l, model) is None]
        if len(unassigned) == 1:
            l = unassigned[0]
            return l.lstrip('~'), not l.startswith('~')
    return None

def dpll(clauses, symbols, model, use_pure=True, use_unit=True):
    global DPLL_CALLS
    DPLL_CALLS += 1
    all_true = True
    for c in clauses:
        v = clause_value(c, model)
        if v is False:
            return False
        if v is None:
            all_true = False
    if all_true:
        return model
    if use_pure:
        p = find_pure_symbol(symbols, clauses, model)
        if p:
            s, val = p
            return dpll(clauses, [x for x in symbols if x != s], {**model, s: val}, use_pure, use_unit)
    if use_unit:
        u = find_unit_clause(clauses, model)
        if u:
            s, val = u
            return dpll(clauses, [x for x in symbols if x != s], {**model, s: val}, use_pure, use_unit)
    s, rest = symbols[0], symbols[1:]
    return (dpll(clauses, rest, {**model, s: True}, use_pure, use_unit) or
            dpll(clauses, rest, {**model, s: False}, use_pure, use_unit))

def clause_symbols(clauses):
    return sorted({l.lstrip('~') for c in clauses for l in c})

def dpll_satisfiable(clauses, use_pure=True, use_unit=True):
    global DPLL_CALLS
    DPLL_CALLS = 0
    clauses = sorted(clauses, key=clause_str)
    return dpll(clauses, clause_symbols(clauses), {}, use_pure, use_unit)

sat_cl = [frozenset(x) for x in (['A'], ['~A','B'], ['~B','C'], ['~C','D','E'], ['~D','~E'])]
m = dpll_satisfiable(sat_cl)
print('Satisfiable:', m, ' calls =', DPLL_CALLS)
unsat_cl = [frozenset(x) for x in (['A','B'], ['~A','B'], ['A','~B'], ['~A','~B'])]
print('Unsatisfiable:', dpll_satisfiable(unsat_cl), ' calls =', DPLL_CALLS)
cl = kb5_clauses | to_clauses(Not(C))
print('CNF(KB & ~C) satisfiable? ->', dpll_satisfiable(cl), ' => KB |= C')

Satisfiable: {'A': True, 'B': True, 'C': True, 'D': True, 'E': False}  calls = 6
Unsatisfiable: False  calls = 5
CNF(KB & ~C) satisfiable? -> False  => KB |= C


## Алхам 7: Wumpus-ийн мэдлэгийн сан

**Шалгах:** KB-ийг хангах загварын тоо 3 байна.
KB ⊨ ¬P12 True, KB ⊨ ¬P21 True, KB ⊨ ¬P22 False, KB ⊨ ¬P31 False, KB ⊨ (P22 ∨
P31) True.
Гурван арга бүх асуулт дээр ижил хариу өгөх ёстой. Зөрөх бол алдаа байна.

In [7]:
P11, P12, P21, P22, P31 = (Sym(n) for n in ('P11','P12','P21','P22','P31'))
B11, B21 = Sym('B11'), Sym('B21')
wumpus_kb = [
    Not(P11),                                   # R1
    Iff(B11, Or(P12, P21)),                     # R2
    Iff(B21, Or(Or(P11, P22), P31)),            # R3
    Not(B11),                                   # R4
    B21,                                        # R5
]
WKB = conjoin(wumpus_kb)
wkb_clauses = set().union(*(to_clauses(s) for s in wumpus_kb))
print('KB clause-ууд:', len(wkb_clauses))
for c in sorted(wkb_clauses, key=clause_str):
    print('  ', clause_str(c))

sat_models = [m for m in all_models(symbols(WKB)) if pl_true(WKB, m)]
print('KB-г хангах загварын тоо:', len(sat_models))
for m in sat_models:
    print('  ', {k: int(v) for k, v in m.items()})

def dpll_entails(kb_clauses, alpha):
    return not dpll_satisfiable(kb_clauses | to_clauses(Not(alpha)))

questions = [Not(P12), Not(P21), Not(P22), Not(P31), Or(P22, P31)]
print()
print(f"{'Асуулт':12s} {'TT':>6s} {'Res':>6s} {'DPLL':>6s}")
for q in questions:
    tt, _ = tt_entails(wumpus_kb, q)
    rs = pl_resolution(wkb_clauses, q)
    dp = dpll_entails(wkb_clauses, q)
    flag = '' if tt == rs == dp else '  <-- ЗӨРӨВ!'
    print(f'{to_str(q):12s} {str(tt):>6s} {str(rs):>6s} {str(dp):>6s}{flag}')

KB clause-ууд: 10
   {B11, ~P12}
   {B11, ~P21}
   {B21, ~P11}
   {B21, ~P22}
   {B21, ~P31}
   {B21}
   {~B11, P12, P21}
   {~B11}
   {~B21, P11, P22, P31}
   {~P11}
KB-г хангах загварын тоо: 3
   {'B11': 0, 'B21': 1, 'P11': 0, 'P12': 0, 'P21': 0, 'P22': 1, 'P31': 1}
   {'B11': 0, 'B21': 1, 'P11': 0, 'P12': 0, 'P21': 0, 'P22': 1, 'P31': 0}
   {'B11': 0, 'B21': 1, 'P11': 0, 'P12': 0, 'P21': 0, 'P22': 0, 'P31': 1}

Асуулт           TT    Res   DPLL
~P12           True   True   True
~P21           True   True   True
~P22          False  False  False
~P31          False  False  False
(P22 | P31)    True   True   True


## Алхам 8: Харьцуулалт

In [8]:
def random_3sat(n, m, seed):
    random.seed(seed)
    clauses = []
    for _ in range(m):
        vars3 = random.sample(range(1, n + 1), 3)
        clauses.append(frozenset(('' if random.random() < 0.5 else '~') + f'x{v}' for v in vars3))
    return clauses

def clauses_to_sentence(clauses):
    def lit(l):
        return Not(Sym(l[1:])) if l.startswith('~') else Sym(l)
    sents = []
    for c in clauses:
        ls = sorted(c)
        s = lit(ls[0])
        for l in ls[1:]:
            s = Or(s, lit(l))
        sents.append(s)
    return conjoin(sents)

def tt_satisfiable(sentence, limit=200_000):
    count = 0
    for m in all_models(symbols(sentence)):
        count += 1
        if count > limit:
            return None, limit
        if pl_true(sentence, m):
            return True, count
    return False, count

def fmt(v):
    return 'хэт удаан' if v is None else v

TT_LIMIT, RES_LIMIT = 200_000, 100_000
rows = []
for q in (Not(P12), Not(P22), Or(P22, P31)):
    ans, tt_n = tt_entails(wumpus_kb, q)
    pl_resolution(wkb_clauses, q, RES_LIMIT); rs_n = RES_COUNT
    dpll_entails(wkb_clauses, q);             dp_n = DPLL_CALLS
    rows.append(('Wumpus: ' + to_str(q), tt_n, rs_n, dp_n, ans))

for n in (10, 15, 20):
    cls = random_3sat(n, 4 * n, seed=1)
    tt, tt_n = tt_satisfiable(clauses_to_sentence(cls), TT_LIMIT)
    rs = pl_resolution(set(cls), None, RES_LIMIT, negate=False); rs_n = RES_COUNT
    model = dpll_satisfiable(cls);                               dp_n = DPLL_CALLS
    rows.append((f'3-SAT, n = {n} (m = {4*n})',
                 None if tt is None else tt_n,
                 None if rs is None else rs_n,
                 dp_n, 'SAT' if model else 'UNSAT'))

print(f"{'Бодлого':24s} {'Үнэний хүснэгт':>15s} {'Резолюц':>12s} {'DPLL':>6s}  Хариу")
for name, tt_n, rs_n, dp_n, ans in rows:
    print(f"{name:24s} {str(fmt(tt_n)):>15s} {str(fmt(rs_n)):>12s} {dp_n:>6}  {ans}")

Бодлого                   Үнэний хүснэгт      Резолюц   DPLL  Хариу
Wumpus: ~P12                         128           10      7  True
Wumpus: ~P22                          93          127      7  False
Wumpus: (P22 | P31)                  128          403      8  True
3-SAT, n = 10 (m = 40)               368    хэт удаан     30  SAT
3-SAT, n = 15 (m = 60)             14770    хэт удаан     98  SAT
3-SAT, n = 20 (m = 80)         хэт удаан    хэт удаан     97  SAT


**Асуултын хариулт**

1. Wumpus дээр DPLL хамгийн бага ажилласан, 7–8 дуудалт л хийсэн. Үнэний хүснэгт 100 орчим загвар шалгасан. Резолюц асуултаас их шалтгаалсан: ¬P12 дээр 10 resolvent гарсан бол (P22 ∨ P31) дээр 403 болсон.

2. Үнэний хүснэгт бүх хослолыг шалгадаг. Хувьсагч нэг нэмэгдэх бүрт хослолын тоо хоёр дахин өснө. n = 20 үед сая гаруй болсон тул хязгаартаа хүрсэн. DPLL бүгдийг шалгадаггүй. Заавал үнэн байх ёстой хувьсагчийг шууд онооно, буруу зам руу орвол тэр дороо буцна. Тиймээс n = 20 дээр ч 97 дуудалт л хийсэн.

3. Резолюц хоосон clause гарсан үед л зогсдог. Бодлого satisfiable бол хоосон clause хэзээ ч гарахгүй. Тиймээс шинэ clause гарахаа болих хүртэл нийлүүлсээр байна. Шинээр гарсан clause-ууд дахиад хоорондоо нийлдэг тул тоо нь маш хурдан өсдөг. Гурван 3-SAT бодлого бүгд дээр резолюц 100 000 хязгаарыг давсан.

## Нэмэлт даалгавар 1: Рыцарь ба луйварчин

**Шалгах:** KB-ийг хангах загвар яг нэг байх ёстой. Тэр загвараар хэн рыцарь, хэн
луйварчин болохыг тодорхойлж, tt_entails-ээр баталгаажуулна (жишээ нь KB ⊨
KA).

In [9]:
KA, KB_, KC = Sym('KA'), Sym('KB'), Sym('KC')
knights_kb = [
    Iff(KA, Not(KB_)),              # A: "B бол луйварчин"
    Iff(KB_, Iff(KA, KC)),          # B: "A ба C ижил төрлийн"
    Iff(KC, KB_),                   # C: "B бол рыцарь"
]
KKB = conjoin(knights_kb)
k_models = [m for m in all_models(symbols(KKB)) if pl_true(KKB, m)]
print('Хангах загварын тоо:', len(k_models), k_models)
for name, s in (('KA', KA), ('~KB', Not(KB_)), ('~KC', Not(KC))):
    print(f'Мэдлэгийн сан |= {name}:', tt_entails(knights_kb, s)[0])

Хангах загварын тоо: 1 [{'KA': True, 'KB': False, 'KC': False}]
Мэдлэгийн сан |= KA: True
Мэдлэгийн сан |= ~KB: True
Мэдлэгийн сан |= ~KC: True


## Нэмэлт даалгавар 2: Unit ба pure literal дүрмийг унтраах

In [10]:
problems = [('Wumpus: ' + to_str(q), list(wkb_clauses | to_clauses(Not(q))))
            for q in (Not(P12), Not(P22), Or(P22, P31))]
problems += [(f'3-SAT, n = {n}', random_3sat(n, 4 * n, 1)) for n in (10, 15, 20)]
print(f"{'Бодлого':24s} {'Хоёулаа':>8s} {'Unit-гүй':>9s} {'Pure-гүй':>9s} {'Хоёулаагүй':>11s}")
for name, cls in problems:
    counts = []
    for up, uu in ((True, True), (True, False), (False, True), (False, False)):
        dpll_satisfiable(cls, use_pure=up, use_unit=uu)
        counts.append(DPLL_CALLS)
    print(f'{name:24s} {counts[0]:>8} {counts[1]:>9} {counts[2]:>9} {counts[3]:>11}')

Бодлого                   Хоёулаа  Unit-гүй  Pure-гүй  Хоёулаагүй
Wumpus: ~P12                    7        10         4           9
Wumpus: ~P22                    7         8         7          11
Wumpus: (P22 | P31)             8        13         8          15
3-SAT, n = 10                  30       122        30         168
3-SAT, n = 15                  98       441        93         858


3-SAT, n = 20                  97      2053        89        2449


**Дүгнэлт:** Гурван арга үргэлж ижил хариу өгсөн. Ялгаа нь зөвхөн хурдад байсан. Үнэний хүснэгтийн хувьд 10 хувьсагчтай үед 1024 хослол, 20 хувьсагчтай үед 1 сая гаруй болж хязгаартаа хүрсэн. Энгийн арга боловч бодлого томрохоор ашиглах боломжгүй болсон. Резолюц 3-SAT бодлого бүгд дээр хязгаар давсан. DPLL бүх бодлого дээр хамгийн хурдан байсан. 20 хувьсагчтай бодлогыг ердөө 97 алхамаар шийдсэн. Үүний гол шалтгаан нь unit propagation байлаа. Надад хамгийн хэцүү санагдсан хэсэг нь CNF хөрвүүлэлтийн distribute функц байсан.